# CorrDiff - Fase 13 - Regimes Meteorológicos

Regimes ERA5 absolutos e anômalos, PCA, KMeans, enriquecimento radar e persistência.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

OUT = Path('../analysis_outputs/13_regimes')
summary = json.loads((OUT/'analysis_summary.json').read_text())
summary


## 1. Diagnósticos de k

In [ ]:
diag = pd.read_parquet(OUT/'clustering_diagnostics.parquet')
display(diag)
for space, g in diag.groupby('regime_space'):
    g = g.sort_values('k')
    plt.figure(figsize=(7,4))
    plt.plot(g.k, g.silhouette, marker='o')
    plt.xlabel('k')
    plt.ylabel('Silhouette')
    plt.title(space)
    plt.tight_layout()
    plt.show()


## 2. Perfis meteorológicos

In [ ]:
profiles = pd.read_parquet(OUT/'regime_predictor_profiles.parquet')
for space in profiles.regime_space.unique():
    for regime in sorted(profiles[profiles.regime_space.eq(space)].regime.unique()):
        t = profiles[(profiles.regime_space.eq(space)) & (profiles.regime.eq(regime))].copy()
        t['abs_anom'] = t.regime_mean_anomaly_sigma.abs()
        display(t.sort_values('abs_anom', ascending=False).head(10)[['predictor','regime_mean','regime_mean_anomaly_sigma']])


## 3. Radar por regime

In [ ]:
radar = pd.read_parquet(OUT/'regime_radar_metrics.parquet')
display(radar)


## 4. Enriquecimento de extremos

In [ ]:
enrich = pd.read_parquet(OUT/'regime_event_enrichment.parquet')
display(enrich[enrich.event_id.isin(['ge_30','ge_40','ge_45'])].sort_values(['regime_space','event_id','risk_ratio_vs_global'], ascending=[True,True,False]))


## 5. Sazonalidade

In [ ]:
season = pd.read_parquet(OUT/'regime_seasonality.parquet')
for space in season.regime_space.unique():
    t = season[season.regime_space.eq(space)]
    display(t.pivot(index='season_code', columns='regime', values='prevalence_regime_within_season'))


## 6. Prevalência anual e 2023–2024

In [ ]:
year = pd.read_parquet(OUT/'regime_yearly_prevalence.parquet')
for space in year.regime_space.unique():
    t = year[(year.regime_space.eq(space)) & (year.year_utc.ge(2020))]
    display(t.pivot(index='year_utc', columns='regime', values='prevalence_within_group'))


## 7. Persistência de 1 hora

In [ ]:
tr = pd.read_parquet(OUT/'regime_transition_1h.parquet')
for space in tr.regime_space.unique():
    t = tr[tr.regime_space.eq(space)]
    display(t.pivot(index='from_regime', columns='to_regime', values='transition_probability').fillna(0))


## 8. Crosswalk absoluto × anomalia

In [ ]:
cw = pd.read_parquet(OUT/'regime_crosswalk.parquet')
display(cw.pivot(index='absolute_regime_id', columns='anomaly_regime_id', values='fraction_within_absolute_regime').fillna(0))


## Regra de interpretação

Os clusters são estados estatísticos ERA5, não classes meteorológicas causais. O radar é usado somente para caracterizar os regimes depois do clustering.